In [3]:
%pip install -q -U transformers torch safetensors huggingface_hub pillow numpy ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ===== Imports =====
from pathlib import Path
from dataclasses import dataclass
from collections import deque, defaultdict
from io import BytesIO
from typing import List, Dict

import time, json, random, itertools
import numpy as np
from PIL import Image

import torch
from transformers import CLIPProcessor, CLIPModel

import ipywidgets as W
from IPython.display import display, clear_output

# ===== Device & Model =====
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "openai/clip-vit-base-patch32"  # D=512

processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

# ===== Utils =====
def l2_normalize(x: np.ndarray, axis=-1, eps=1e-12):
    """L2-normalize array along a given axis."""
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(n, eps)

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two unit vectors (1D)."""
    return float(np.dot(a, b))

DWELL_THRESHOLD_SECONDS = 3.0
SCORE_LIKE, SCORE_SWIPE_FAST, SCORE_SWIPE_SLOW = 1.0, 0.0, 0.5

def event_to_feedback(action: str, dwell_seconds: float | None):
    """Map a UI event to a numeric feedback signal."""
    if action == "like":
        return SCORE_LIKE
    if action == "swipe":
        return SCORE_SWIPE_SLOW if (dwell_seconds or 0) >= DWELL_THRESHOLD_SECONDS else SCORE_SWIPE_FAST
    return 0.0

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# ===== Project paths (adjust if needed) =====
PROJECT_ROOT = Path.cwd().resolve().parents[1]
CANDIDATE_DIRS = [
    PROJECT_ROOT / "Unsplash",
    PROJECT_ROOT / "images",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,  # fallback
]

def find_images(dirs, limit=20000):
    """Recursively collect image paths up to `limit`."""
    exts = ("*.jpg","*.jpeg","*.png","*.webp")
    files = []
    for d in dirs:
        if not Path(d).exists():
            continue
        for ext in exts:
            files += list(Path(d).rglob(ext))
    files = [str(p) for p in files]
    random.shuffle(files)
    return files[:limit]

paths = find_images(CANDIDATE_DIRS, limit=20000)
print(f"Found images: {len(paths)}")
assert len(paths) > 0, "No images found. Update CANDIDATE_DIRS."

# ===== Quality heuristics (lightweight, no OpenCV) =====
def _edge_energy(im: Image.Image) -> float:
    """
    Cheap sharpness proxy:
    - Convert to grayscale 128x128
    - Sum mean absolute horizontal/vertical gradients
    """
    g = im.convert("L").resize((128,128), Image.BILINEAR)
    arr = np.asarray(g, dtype=np.float32)
    dx = np.abs(np.diff(arr, axis=1)).mean()
    dy = np.abs(np.diff(arr, axis=0)).mean()
    return float(dx + dy)

def _ahash(im: Image.Image, size=8) -> int:
    """Average hash (aHash) for exact/near-exact duplicates; returns a 64-bit int."""
    g = im.convert("L").resize((size, size), Image.BILINEAR)
    arr = np.asarray(g, dtype=np.float32)
    thr = arr.mean()
    bits = (arr > thr).astype(np.uint8).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return int(h)

@dataclass
class ImageMeta:
    path: str
    width: int
    height: int
    edge_energy: float
    ahash: int

def inspect_images(paths: List[str]) -> List[ImageMeta]:
    """Read small info once; ignore unreadable images silently."""
    out = []
    for p in paths:
        try:
            im = Image.open(p).convert("RGB")
            w, h = im.size
            ee = _edge_energy(im)
            ah = _ahash(im)
            out.append(ImageMeta(p, w, h, ee, ah))
        except Exception:
            continue
    return out

meta = inspect_images(paths)
paths = [m.path for m in meta]  # align after skipping any failures
print(f"Inspected: {len(meta)} (kept)")

# Build duplicate groups by aHash
hash2idxs: Dict[int, List[int]] = defaultdict(list)
for i, m in enumerate(meta):
    hash2idxs[m.ahash].append(i)
dupe_groups = [v for v in hash2idxs.values() if len(v) > 1]
print(f"Exact-dupe groups (aHash): {len(dupe_groups)}")


Found images: 5101
Inspected: 5101 (kept)
Exact-dupe groups (aHash): 259


In [8]:
# ===== Batch CLIP embeddings with caching =====
EMB_DIR = Path("./emb_cache")
EMB_DIR.mkdir(exist_ok=True)
E_PATH = EMB_DIR / "E_fp32.npy"
P_PATH = EMB_DIR / "paths.npy"
M_PATH = EMB_DIR / "meta.json"   # optional

def get_image_features_batch(batch_imgs: List[Image.Image]) -> np.ndarray:
    """Encode a batch of PIL images to unit CLIP image features (float32)."""
    inputs = processor(images=batch_imgs, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs).float()
    emb = emb.cpu().numpy()
    emb = l2_normalize(emb, axis=1).astype(np.float32)
    return emb

def encode_all(paths: List[str], batch_size=32) -> np.ndarray:
    """Memory-safe batching with tiny thumbnails handled by CLIP processor."""
    E_list, batch = [], []
    for i, p in enumerate(paths, 1):
        try:
            batch.append(Image.open(p).convert("RGB"))
        except Exception:
            batch.append(Image.new("RGB",(224,224),(0,0,0)))
        if len(batch) == batch_size or i == len(paths):
            E_list.append(get_image_features_batch(batch))
            batch.clear()
        if i % 200 == 0:
            print(f"encoded {i}/{len(paths)}")
    E = np.vstack(E_list).astype(np.float32)
    E = l2_normalize(E, axis=1)
    return E

def save_embeddings(E: np.ndarray, paths: List[str]):
    np.save(E_PATH, E)
    np.save(P_PATH, np.array(paths))

def load_embeddings():
    if E_PATH.exists() and P_PATH.exists():
        E = np.load(E_PATH)
        p = np.load(P_PATH, allow_pickle=True).tolist()
        return E, p, True
    return None, None, False

E, paths2, loaded = load_embeddings()
if loaded and len(paths2) == len(paths) and paths2 == paths:
    print("Loaded embeddings from cache.")
else:
    print("Computing embeddings from scratch...")
    E = encode_all(paths, batch_size=32)
    save_embeddings(E, paths)
    print("Saved embeddings to emb_cache/")

# Keep metadata arrays aligned to `paths`
idx_map = {m.path: i for i, m in enumerate(meta)}
meta = [meta[idx_map[p]] for p in paths]
E = l2_normalize(E, axis=1).astype(np.float32)
N, D = E.shape
print("E shape:", E.shape)

# ===== Runtime masks and guards (updated live by the panel) =====
# Defaults (can be changed from the UI)
QUALITY_MIN_EDGE = 3.0         # min edge energy
QUALITY_MIN_W, QUALITY_MIN_H = 256, 256
HIDE_EXACT_DUPES = True
DIVERSITY_LAST_K = 10
DIVERSITY_MIN_COS = 0.95       # don't show if similarity >= this to any of last-K

# Precompute basic quality mask
def compute_quality_mask(min_edge, min_w, min_h):
    """Basic size/sharpness mask from metadata only."""
    q = np.ones(N, dtype=bool)
    for i, m in enumerate(meta):
        if m.edge_energy < min_edge or m.width < min_w or m.height < min_h:
            q[i] = False
    return q

# Map each index to its aHash dupes group; used to hide exact duplicates
ahash_groups = {i: set() for i in range(N)}
for group in dupe_groups:
    s = set(group)
    for i in group:
        ahash_groups[i] = s


Computing embeddings from scratch...
encoded 200/5101
encoded 400/5101
encoded 600/5101
encoded 800/5101
encoded 1000/5101
encoded 1200/5101
encoded 1400/5101
encoded 1600/5101
encoded 1800/5101
encoded 2000/5101
encoded 2200/5101
encoded 2400/5101
encoded 2600/5101
encoded 2800/5101
encoded 3000/5101
encoded 3200/5101
encoded 3400/5101
encoded 3600/5101
encoded 3800/5101
encoded 4000/5101
encoded 4200/5101
encoded 4400/5101
encoded 4600/5101
encoded 4800/5101
encoded 5000/5101
Saved embeddings to emb_cache/
E shape: (5101, 512)


In [1]:
# ================= TAGS (Explain-only, 100 labels) =================
label_vocab = [
    # 1-10 transport / luxury
    "supercars","sports cars","classic cars","luxury cars","off-road trucks",
    "motorcycles","motorsport","yacht","private jet","racing car",
    # 11-22 watches / jewelry / fashion
    "luxury watches","mechanical watch","chronograph","jewelry","diamonds",
    "rings","necklaces","handbags","sneakers","menswear","womenswear","streetwear",
    # 23-28 portraits / people
    "portraits","studio portrait","headshot","close-up face","street photography","fashion photography",
    # 29-36 architecture / interiors / hotels
    "architecture","modern interiors","luxury interiors","minimal interiors",
    "skyscrapers","luxury hotel","resort","spa",
    # 37-47 nature / landscapes / sky
    "landscapes","mountains","beaches","desert","forests","waterfalls",
    "sunsets","night sky","milky way","aurora","aerial",
    # 48-57 wildlife / animals
    "wildlife","dogs","cats","birds","horses","lions","tigers","elephants","wolves","foxes",
    # 58-67 city / travel
    "cityscapes","old town","alleyway","night city","street market",
    "train station","airport lounge","harbor","bridge","tower",
    # 68-80 food & drinks
    "food","desserts","coffee","latte art","tea","sushi","pizza","burgers",
    "steak","pasta","salad","breakfast","fine dining",
    # 81-90 sports & fitness
    "football","basketball","tennis","boxing","golf","gym fitness",
    "cycling","running","swimming","skiing",
    # 91-98 tech & gadgets
    "technology","gadgets","smartphones","laptops","gaming setup","workstation","headphones","camera gear",
    # 99-100 art styles
    "abstract art","minimalism"
]
assert len(label_vocab) == 100

TEMPLATES = [
    "a photo of {}",
    "a high quality photo of {}",
    "aesthetic {}",
    "premium {}",
    "close-up of {}",
    "{}"
]

TXT_EMB_PATH = EMB_DIR / "txt_emb_tags100.npy"
TXT_LABS_PATH= EMB_DIR / "txt_labels_tags100.json"

def build_or_load_text_embeddings():
    if TXT_EMB_PATH.exists() and TXT_LABS_PATH.exists():
        txt_emb = np.load(TXT_EMB_PATH)
        labs = json.loads(TXT_LABS_PATH.read_text())
        if labs == label_vocab and txt_emb.shape[0] == len(label_vocab):
            return txt_emb
    with torch.no_grad():
        rows = []
        for lab in label_vocab:
            prompts = [t.format(lab) for t in TEMPLATES]
            tin = processor(text=prompts, return_tensors="pt", padding=True).to(device)
            tfeat = model.get_text_features(**tin).float().cpu().numpy().mean(axis=0)
            rows.append(tfeat)
    txt_emb = l2_normalize(np.stack(rows, axis=0), axis=1).astype(np.float32)
    np.save(TXT_EMB_PATH, txt_emb)
    TXT_LABS_PATH.write_text(json.dumps(label_vocab, ensure_ascii=False))
    return txt_emb

txt_emb = build_or_load_text_embeddings()

def explain_trending_tags(E_unit: np.ndarray, pref: np.ndarray, top_pool=300, k=10, calibrate=True):
    """
    Explanation only (not used for ranking):
    - Take `top_pool` images nearest to `pref` and compute centroid.
    - Project centroid onto text-label embeddings to get top tags.
    - Optionally subtract dataset baseline to reduce common-category bias.
    """
    sims = E_unit @ pref
    kpool = min(top_pool, np.isfinite(sims).sum())
    if kpool <= 0: return []
    I = np.argpartition(-sims, range(kpool))[:kpool]
    pool_centroid = l2_normalize(E_unit[I].mean(axis=0))
    scores = txt_emb @ pool_centroid
    if calibrate:
        base = l2_normalize(E_unit.mean(axis=0))
        scores -= (txt_emb @ base)
    J = np.argsort(-scores)[:k]
    return [(label_vocab[j], float(scores[j])) for j in J]


NameError: name 'EMB_DIR' is not defined

In [ ]:
# ================================================================
# ===== ImageRecommender v4 (clean install) =====
import numpy as np
from collections import deque

def event_to_feedback(action: str, dwell_seconds: float | None):
    """Linear: fast swipe ~ -0.25 .. linger 3s ~ +0.5 ; like = +1.0"""
    if action == "like":
        return 1.0
    if action == "swipe":
        d = 0.0 if dwell_seconds is None else float(dwell_seconds)
        w = max(0.0, min(d / 3.0, 1.0))     # 0..1
        return -0.25 + 0.75 * w             # [-0.25 .. +0.5]
    return 0.0

def l2_normalize(x: np.ndarray, axis=-1, eps=1e-12):
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(n, eps)

class ImageRecommender:
    def __init__(self, E_unit: np.ndarray, quality_mask: np.ndarray | None = None,
                 alpha=0.85, eta=1.6, warmup_n=3,
                 recent_k=50, recent_weight=0.8, focus_gamma=1.3,
                 diversity_last_k=20, diversity_min_cos=0.92,
                 hide_exact_dupes=True, rng=None):
        self.E = l2_normalize(np.asarray(E_unit), axis=1)
        self.N, self.D = self.E.shape

        self.alpha = float(alpha); self.eta = float(eta)
        self.warmup_n = int(warmup_n); self.focus_gamma = float(focus_gamma)
        self.recent_k = int(recent_k); self.recent_weight = float(recent_weight)
        self.diversity_last_k = int(diversity_last_k); self.diversity_min_cos = float(diversity_min_cos)
        self.hide_exact_dupes = bool(hide_exact_dupes)

        self.rng = rng or np.random.default_rng(42)
        base = self.E.mean(axis=0) + self.rng.normal(0.0, 0.05, size=self.D)
        self.preference = l2_normalize(base).astype(np.float32)

        self.quality_mask = quality_mask if quality_mask is not None else np.ones(self.N, bool)
        self.seen = set()
        self._last_shown = deque(maxlen=self.diversity_last_k)
        self._recent = deque(maxlen=self.recent_k)

        self._warm_count = 0; self._warm_sum = np.zeros(self.D, np.float32)
        self._warm_weight = 0.0; self._warmed_up = (self.warmup_n <= 0)
        self._warm_order = self.rng.permutation(self.N); self._warm_ptr = 0

    def set_quality_mask(self, mask: np.ndarray):
        self.quality_mask = mask

    def set_diversity(self, last_k: int, min_cos: float):
        self.diversity_last_k = int(last_k); self.diversity_min_cos = float(min_cos)
        self._last_shown = deque(list(self._last_shown), maxlen=self.diversity_last_k)

    def _passes_diversity(self, idx: int) -> bool:
        if not self._last_shown or self.diversity_min_cos >= 0.999:
            return True
        e = self.E[idx]
        for li in self._last_shown:
            if float(np.dot(e, self.E[li])) >= self.diversity_min_cos:
                return False
        return True

    def _mask_candidates(self) -> np.ndarray:
        mask = self.quality_mask.copy()
        if self.seen: mask[list(self.seen)] = False
        if self.hide_exact_dupes and self.seen:
            try:
                banned = set()
                for i in self.seen:
                    banned |= ahash_groups.get(i, set())
                if banned:
                    mask[list(banned)] = False
            except NameError:
                pass
        return mask

    def _pick_warmup(self):
        mask = self._mask_candidates()
        while self._warm_ptr < self.N:
            idx = int(self._warm_order[self._warm_ptr]); self._warm_ptr += 1
            if not mask[idx]:
                continue
            if not self._passes_diversity(idx):
                continue
            return idx
        return None

    def update_with_event(self, idx: int, action: str, dwell_seconds: float | None):
        fb = event_to_feedback(action, dwell_seconds)
        self._update(idx, fb)

    def _update(self, idx: int, feedback: float):
        e = self.E[idx]; self.seen.add(idx); self._last_shown.append(idx)

        # warm-up
        if not self._warmed_up:
            self._warm_count += 1
            if feedback > 0:
                self._warm_sum += feedback * e
                self._warm_weight += feedback
            if (self._warm_count >= self.warmup_n) or (self._warm_weight >= 1.0):
                if self._warm_weight > 0:
                    delta = self._warm_sum / self._warm_weight
                    self.preference = l2_normalize(self.alpha * self.preference + self.eta * delta)
                self._warmed_up = True; self._warm_sum[:] = 0; self._warm_weight = 0.0
            return

        # online EMA
        if feedback > 0:
            self._recent.append(feedback * e)
        if self.recent_weight > 0 and len(self._recent) > 0:
            recent_centroid = np.mean(np.stack(self._recent, axis=0), axis=0)
        else:
            recent_centroid = 0.0
        update_vec = self.alpha * self.preference + self.eta * feedback * e + self.recent_weight * recent_centroid
        self.preference = l2_normalize(update_vec)

    def recommend_next(self):
        if len(self.seen) >= self.N: return None, None

        if not self._warmed_up:
            idx = self._pick_warmup()
            return (idx, None) if idx is not None else (None, None)

        sims = self.E @ self.preference
        mask = self._mask_candidates()
        sims = np.where(mask, sims, -np.inf)

        g = self.focus_gamma
        if g != 1.0:
            s = np.clip(sims, -1.0, 1.0)
            sims = np.sign(s) * (np.abs(s) ** g)

        order = np.argsort(-sims)
        for i in order:
            if not np.isfinite(sims[i]): break
            if self._passes_diversity(int(i)):
                return int(i), float(sims[i])
        return None, None

    def recommend_next_mmr(self, pool_k=200, lambda_=0.7):
        if len(self.seen) >= self.N: return None, None

        if not self._warmed_up:
            idx = self._pick_warmup()
            return (idx, None) if idx is not None else (None, None)

        sims = self.E @ self.preference
        mask = self._mask_candidates()
        sims = np.where(mask, sims, -np.inf)

        k = min(pool_k, np.isfinite(sims).sum())
        if k <= 0: return None, None
        pool = np.argpartition(-sims, range(k))[:k]

        if not len(self._last_shown):
            j = int(pool[np.argmax(sims[pool])])
            return j, float(sims[j])

        Epool = self.E[pool]
        sim_pref = sims[pool]
        sim_to_sel = Epool @ self.E[list(self._last_shown)].T
        max_sim_sel = sim_to_sel.max(axis=1)

        mmr = lambda_ * sim_pref - (1 - lambda_) * max_sim_sel
        j_local = int(np.argmax(mmr))
        j = int(pool[j_local])
        return j, float(sim_pref[j_local])

# ===== Instantiate a fresh recommender =====
try:
    quality_mask = compute_quality_mask(min_edge=2.0, min_w=256, min_h=256)
except NameError:
    quality_mask = np.ones(E.shape[0], dtype=bool)

rec = ImageRecommender(
    E,
    quality_mask=quality_mask,
    alpha=0.85, eta=1.6, warmup_n=3,
    recent_k=50, recent_weight=0.8,
    focus_gamma=1.3,
    diversity_last_k=20, diversity_min_cos=0.92,
    hide_exact_dupes=True
)

print("Recommender ready:", rec.N, "items; D=", rec.D)


Recommender ready: 5101 items; D= 512


In [2]:
# ============== Quality Filter v2 (zoom/cutout/semantics) ==============
import numpy as np, json, time
from pathlib import Path
from PIL import Image

EMB_DIR = Path("./emb_cache"); EMB_DIR.mkdir(exist_ok=True)
QF_META = EMB_DIR / "quality_meta_v2.npz"
NEG_EMB = EMB_DIR / "neg_txt_emb.npy"
NEG_LABS= EMB_DIR / "neg_txt_labels.json"

# Negative prompts (things we want to suppress) — short & clear
NEG_PROMPTS = [
    "isolated object on white background",
    "isolated animal on white background",
    "product cutout on white background",
    "plain solid background",
    "macro close-up crop",
    "extreme zoom crop",
    "texture pattern close-up",
    "blurry out of focus photo",
    "logo or icon or clipart",
    "screenshot of user interface",
    "meme with big text",
    "watermark text overlay"
]

def _build_or_load_neg_text_emb():
    if NEG_EMB.exists() and NEG_LABS.exists():
        txt = np.load(NEG_EMB)
        labs = json.loads(NEG_LABS.read_text())
        if labs == NEG_PROMPTS:
            return txt
    rows = []
    with torch.no_grad():
        for lab in NEG_PROMPTS:
            prompts = [
                f"a photo of {lab}",
                f"high quality {lab}",
                f"{lab}"
            ]
            tin = processor(text=prompts, return_tensors="pt", padding=True).to(device)
            tfeat = model.get_text_features(**tin).float().cpu().numpy().mean(axis=0)
            rows.append(tfeat)
    neg_txt_emb = l2_normalize(np.stack(rows, axis=0), axis=1).astype(np.float32)
    np.save(NEG_EMB, neg_txt_emb); NEG_LABS.write_text(json.dumps(NEG_PROMPTS))
    return neg_txt_emb

_neg_txt_emb = _build_or_load_neg_text_emb()

def _quick_features_for_path(p, thumb=128):
    """
    Return: (w,h, edge_energy, center_edge_ratio, white_bg_frac, has_alpha_cutout)
    """
    try:
        im = Image.open(p)
        has_alpha = (im.mode in ("LA","RGBA","PA"))
        alpha_cut = False
        if has_alpha:
            im = im.convert("RGBA")
            a = np.asarray(im)[:,:,3].astype(np.float32)/255.0
            alpha_cut = (a < 0.05).mean() > 0.10  # >10% transparent
            im = im.convert("RGB")
        else:
            im = im.convert("RGB")
        w, h = im.size
        im2 = im.copy()
        im2.thumbnail((thumb, thumb))
        arr = np.asarray(im2).astype(np.float32)/255.0
        gray = (0.299*arr[:,:,0] + 0.587*arr[:,:,1] + 0.114*arr[:,:,2])

        # simple gradient magnitude
        gx = np.abs(np.diff(gray, axis=1, prepend=gray[:,[0]]))
        gy = np.abs(np.diff(gray, axis=0, prepend=gray[[0],:]))
        edge = gx + gy
        edge_energy = float(edge.mean()*100.0)

        # center vs border ratio
        H,W = gray.shape
        m1,m2 = int(H*0.25), int(H*0.75)
        n1,n2 = int(W*0.25), int(W*0.75)
        center = edge[m1:m2, n1:n2].mean() + 1e-8
        border = np.concatenate([
            edge[:m1,:], edge[m2:,:], edge[m1:m2,:n1], edge[m1:m2,n2:]
        ], axis=None).mean() + 1e-8
        center_edge_ratio = float(center / border)  # larger = zoomed center

        # white/solid background fraction
        std = arr.std(axis=2)
        white = (arr.mean(axis=2) > 0.92) & (std < 0.03)
        white_bg_frac = float(white.mean())

        return w, h, edge_energy, center_edge_ratio, white_bg_frac, alpha_cut
    except Exception:
        return 0,0,0.0, 1.0, 0.0, False

def _build_or_load_qf_meta(paths):
    if QF_META.exists():
        dat = np.load(QF_META, allow_pickle=False)
        if int(dat["N"]) == len(paths):
            return {k: dat[k] for k in dat.files}
    feats = [ _quick_features_for_path(p) for p in paths ]
    w,h,ee,cer,wbg,ac = map(np.array, zip(*feats))
    np.savez_compressed(QF_META, N=len(paths), w=w, h=h, edge=ee, center_ratio=cer, whitebg=wbg, alpha_cut=ac)
    return {"N":len(paths), "w":w, "h":h, "edge":ee, "center_ratio":cer, "whitebg":wbg, "alpha_cut":ac}

_qf = _build_or_load_qf_meta(paths)

def compute_quality_mask_v2(min_edge=2.0, min_w=256, min_h=256,
                            detect_zoom=True,  zoom_center_ratio=1.8,
                            detect_cutout=True, solid_bg_frac=0.60, alpha_frac=0.10,
                            use_negative_semantics=True, weird_thresh=0.32):
    """
    Return boolean quality mask (True = keep).
    - detect_zoom: reject if center_edge_ratio too high (over-zoom).
    - detect_cutout: reject solid white BG / large alpha cutout.
    - negative semantics: reject if too similar to any NEG_PROMPTS.
    """
    w = _qf["w"]; h=_qf["h"]; edge=_qf["edge"]; cer=_qf["center_ratio"]; wbg=_qf["whitebg"]; ac=_qf["alpha_cut"].astype(np.float32)

    ok = (w>=min_w) & (h>=min_h) & (edge>=min_edge)

    if detect_zoom:
        ok &= (cer <= zoom_center_ratio)

    if detect_cutout:
        cut = (wbg >= solid_bg_frac) | (ac >= alpha_frac)
        ok &= ~cut

    if use_negative_semantics:
        sims = E @ _neg_txt_emb.T   # (N, M)
        worst = sims.max(axis=1)    # highest similarity to any negative prompt
        ok &= (worst < weird_thresh)

    return ok.astype(bool)


NameError: name 'torch' is not defined

In [ ]:
# C8 — Interactive Control Panel (fixed + per-action analysis + user orientation)
import ipywidgets as W
from IPython.display import display, clear_output
from time import perf_counter
import numpy as np
from PIL import Image
from pathlib import Path

# ---------- compatibility guard (old rec objects) ----------
if not hasattr(rec, "_updates_total"):
    rec._updates_total = 0

# ---------- explain-only helpers ----------
def image_labels(idx: int, top_k: int = 6):
    e = rec.E[idx]                    # unit image embedding
    scores = (txt_emb @ e)            # (num_labels,)
    J = np.argsort(-scores)[:top_k]
    return [(label_vocab[j], float(scores[j])) for j in J]

def user_orientation(top_k: int = 8):
    scores = (txt_emb @ rec.preference)   # projection of user vector
    J = np.argsort(-scores)[:top_k]
    return [(label_vocab[j], float(scores[j])) for j in J]

# ---------- UI controls ----------
# Quality
min_edge = W.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description='min_edge')
min_w    = W.IntSlider(value=256, min=128, max=2048, step=64, description='min_w')
min_h    = W.IntSlider(value=256, min=128, max=2048, step=64, description='min_h')
detect_zoom = W.Checkbox(value=True, description='detect_zoom')
zoom_center_ratio = W.FloatSlider(value=1.8, min=1.0, max=3.0, step=0.05, description='zoom_ratio')
detect_cutout = W.Checkbox(value=True, description='detect_cutout')
solid_bg_frac = W.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.05, description='solid_bg')
alpha_frac    = W.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description='alpha_frac')
use_neg = W.Checkbox(value=True, description='neg_semantics')
weird_thresh = W.FloatSlider(value=0.32, min=0.1, max=0.7, step=0.02, description='neg_thresh')

quality_box = W.VBox([
    W.HTML("<b>Quality filters</b>"),
    W.HBox([min_edge, min_w, min_h]),
    W.HBox([detect_zoom, zoom_center_ratio]),
    W.HBox([detect_cutout, solid_bg_frac, alpha_frac]),
    W.HBox([use_neg, weird_thresh]),
])

# Model params
alpha = W.FloatSlider(value=0.85, min=0.50, max=0.99, step=0.01, description='alpha')
eta   = W.FloatSlider(value=1.60, min=0.05, max=3.00, step=0.05, description='eta')
recent_weight = W.FloatSlider(value=0.80, min=0.00, max=1.50, step=0.05, description='recent_w')
focus_gamma   = W.FloatSlider(value=1.30, min=1.00, max=2.00, step=0.01, description='focus_γ')

model_box = W.VBox([
    W.HTML("<b>Model params</b>"),
    W.HBox([alpha, eta, recent_weight, focus_gamma]),
])

# Diversity + dupes
div_last_k = W.IntSlider(value=20, min=0, max=50, step=1, description='last_k')
div_mincos = W.FloatSlider(value=0.92, min=0.80, max=0.99, step=0.01, description='min_cos')
hide_dupes = W.Checkbox(value=True, description='hide_exact_dupes')

div_box = W.VBox([
    W.HTML("<b>Diversity</b>"),
    W.HBox([div_last_k, div_mincos, hide_dupes]),
])

# Stability-after-N
enable_stab = W.Checkbox(value=True, description='enable_stability')
stab_after  = W.IntSlider(value=20, min=1, max=100, step=1, description='after_N')
min_eta     = W.FloatSlider(value=0.05, min=0.01, max=0.5, step=0.01, description='min_eta')
min_recent  = W.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.02, description='min_recent')
gamma_cap   = W.FloatSlider(value=1.60, min=1.0, max=2.0, step=0.01, description='γ_cap')
decay       = W.FloatSlider(value=0.95, min=0.80, max=0.99, step=0.01, description='decay')
focus_step  = W.FloatSlider(value=0.01, min=0.0, max=0.05, step=0.005, description='γ_step')

stab_box = W.VBox([
    W.HTML("<b>Stability after N interactions</b>"),
    W.HBox([enable_stab, stab_after, min_eta, min_recent]),
    W.HBox([gamma_cap, decay, focus_step]),
])

# Warm-up controls
warmup_n = W.IntSlider(value=max(3, getattr(rec, "warmup_n", 3)), min=0, max=50, step=1, description='warmup_n')
diverse_warmup = W.Checkbox(value=True, description='diverse warmup')
build_warmup = W.Button(description="Build warmup order")

warmup_box = W.VBox([
    W.HTML("<b>Warm-up (first images)</b>"),
    W.HBox([warmup_n, diverse_warmup, build_warmup]),
])

# Buttons (Like / Next / Reset)
btn_apply = W.Button(description="Apply filters & params", button_style='info')
btn_next  = W.Button(description="Next ▶")
btn_like  = W.Button(description="Like (+1) 👍", button_style='success')
btn_reset = W.Button(description="Reset user", button_style='warning')

buttons = W.HBox([btn_apply, btn_next, btn_like, btn_reset])

# Output area
out = W.Output(layout={'border': '1px solid #ddd', 'padding': '6px'})

# ---------- runtime helpers ----------
_last_t0 = None
_current_idx = None
DWELL_THRESHOLD = 3.0        # seconds
FB_LONG_DWELL   = +0.5
FB_SHORT_DWELL  = -0.15

STATE_DIR = Path("./session_state"); STATE_DIR.mkdir(exist_ok=True)

def _apply_params_to_model():
    rec.alpha = float(alpha.value)
    rec.eta = float(eta.value)
    rec.recent_weight = float(recent_weight.value)
    rec.focus_gamma = float(focus_gamma.value)
    rec.set_diversity(int(div_last_k.value), float(div_mincos.value))
    rec.hide_exact_dupes = bool(hide_dupes.value)
    rec.warmup_n = int(warmup_n.value)

def _recompute_quality():
    q = compute_quality_mask_v2(
        min_edge=float(min_edge.value),
        min_w=int(min_w.value), min_h=int(min_h.value),
        detect_zoom=bool(detect_zoom.value), zoom_center_ratio=float(zoom_center_ratio.value),
        detect_cutout=bool(detect_cutout.value), solid_bg_frac=float(solid_bg_frac.value), alpha_frac=float(alpha_frac.value),
        use_negative_semantics=bool(use_neg.value), weird_thresh=float(weird_thresh.value),
    )
    rec.set_quality_mask(q)
    return int(q.sum())

def _enable_stability_patch():
    # patch only once
    if getattr(rec, "_stability_patched", False): return
    rec._stability_patched = True
    if not hasattr(rec, "_original_update"):
        rec._original_update = rec._update
    def _update_with_stability(idx, feedback):
        # call original update
        before = getattr(rec, "_updates_total", 0)
        rec._original_update(idx, feedback)
        after = getattr(rec, "_updates_total", before)
        if after == before:
            rec._updates_total = before + 1  # ensure counter exists even for old class versions
        # schedule
        if enable_stab.value and rec._updates_total >= int(stab_after.value):
            rec.eta = max(rec.eta * float(decay.value), float(min_eta.value))
            rec.recent_weight = max(rec.recent_weight * float(decay.value), float(min_recent.value))
            rec.focus_gamma = min(rec.focus_gamma + float(focus_step.value), float(gamma_cap.value))
    rec._update = _update_with_stability

def _build_diverse_warmup(k: int):
    mask = rec._mask_candidates()
    I = np.where(mask)[0]
    if I.size == 0: return
    POOL = I if I.size <= 1500 else np.random.default_rng(0).choice(I, size=1500, replace=False)
    Epool = rec.E[POOL]
    rng = np.random.default_rng(0)
    sel = [int(rng.integers(low=0, high=len(POOL)))]
    while len(sel) < min(k, len(POOL)):
        sims = Epool @ Epool[sel].T
        max_sim = sims.max(axis=1)
        cand = int(np.argmin(max_sim))
        if cand in sel: break
        sel.append(cand)
    warm = POOL[sel].tolist()
    rest = [int(x) for x in POOL if int(x) not in warm]
    rng.shuffle(rest)
    order = warm + rest + [int(x) for x in I if int(x) not in set(POOL)]
    rec._warm_order = np.array(order, dtype=np.int64)
    rec._warm_ptr = 0

def _render(idx, note=""):
    global _last_t0, _current_idx
    _current_idx = idx
    _last_t0 = perf_counter()
    p = paths[idx]
    sim = float(np.dot(rec.E[idx], rec.preference))
    labels = image_labels(idx, top_k=6)
    orient = user_orientation(top_k=6)
    # display image
    im = Image.open(p).convert("RGB")
    max_side = 720
    w,h = im.size
    s = min(max_side/w, max_side/h, 1.0)
    if s < 1.0: im = im.resize((int(w*s), int(h*s)))
    with out:
        clear_output(wait=True)
        display(im)
        print(f"idx={idx} | sim={sim:.3f} | seen={len(rec.seen)} / {rec.N} | {note}")
        print("Image labels:", ", ".join([t for t,_ in labels]))
        print("User orientation:", ", ".join([t for t,_ in orient]))

def _next(apply_prev_feedback: bool = True):
    # commit dwell feedback on current image
    if apply_prev_feedback and _current_idx is not None:
        dwell = perf_counter() - _last_t0 if _last_t0 is not None else 0.0
        fb = FB_LONG_DWELL if dwell >= DWELL_THRESHOLD else FB_SHORT_DWELL
        sim_before = float(np.dot(rec.E[int(_current_idx)], rec.preference))
        rec._update(int(_current_idx), float(fb))
        sim_after = float(np.dot(rec.E[int(_current_idx)], rec.preference))
        note = f"NEXT: dwell={dwell:.2f}s → feedback={fb:+.2f} | sim {sim_before:.3f}→{sim_after:.3f}"
    else:
        note = "NEXT"
    idx, _ = rec.recommend_next_mmr(pool_k=200, lambda_=0.7)
    if idx is None:
        with out:
            clear_output(wait=True)
            print("No candidate found. Relax filters or press Reset user.")
        return
    _render(idx, note=note)

def _like():
    if _current_idx is None:
        _next(apply_prev_feedback=False); return
    sim_before = float(np.dot(rec.E[int(_current_idx)], rec.preference))
    rec._update(int(_current_idx), 1.0)  # +1
    sim_after = float(np.dot(rec.E[int(_current_idx)], rec.preference))
    note = f"LIKE: feedback=+1.00 | sim {sim_before:.3f}→{sim_after:.3f}"
    idx, _ = rec.recommend_next_mmr(pool_k=200, lambda_=0.7)
    if idx is None:
        with out:
            clear_output(wait=True)
            print("End of candidates. Try Reset user.")
        return
    _render(idx, note=note)

def _reset_user():
    rec.seen.clear(); rec._last_shown.clear(); rec._recent.clear()
    rec._warm_count = 0; rec._warm_sum[:] = 0; rec._warm_weight = 0.0
    rec._warmed_up = (rec.warmup_n <= 0); rec._warm_ptr = 0
    rec._updates_total = 0
    base = rec.E.mean(axis=0) + np.random.default_rng(42).normal(0.0, 0.05, size=rec.D)
    rec.preference = l2_normalize(base).astype(np.float32)
    with out:
        clear_output(wait=True)
        print("User state reset. Press Next to start.")
    _next(apply_prev_feedback=False)

# ---------- callbacks ----------
def on_apply_clicked(_):
    kept = _recompute_quality()
    _apply_params_to_model()
    _enable_stability_patch()
    with out:
        clear_output(wait=True)
        print(f"Applied. Kept {kept} / {len(rec.quality_mask)} | warmup_n={rec.warmup_n} | stability_after={stab_after.value}")

btn_apply.on_click(on_apply_clicked)
btn_next.on_click(lambda _: _next(apply_prev_feedback=True))
btn_like.on_click(lambda _: _like())
btn_reset.on_click(lambda _: _reset_user())

def on_build_warmup(_):
    if bool(diverse_warmup.value):
        _build_diverse_warmup(int(warmup_n.value))
    else:
        mask = rec._mask_candidates()
        I = np.where(mask)[0]
        order = np.array(np.random.default_rng(0).permutation(I), dtype=np.int64)
        rec._warm_order = order; rec._warm_ptr = 0
    with out:
        clear_output(wait=True)
        print(f"Warm-up order prepared. diverse={bool(diverse_warmup.value)} | k={int(warmup_n.value)}")
build_warmup.on_click(on_build_warmup)

# ---------- layout ----------
ui = W.VBox([
    W.HBox([quality_box, model_box]),
    W.HBox([div_box, stab_box]),
    warmup_box,
    buttons,
    out
])
display(ui)

# Initialize and show first candidate
_recompute_quality()
_apply_params_to_model()
_enable_stability_patch()
on_build_warmup(None)
_next(apply_prev_feedback=False)
